In [ ]:
# load in adata. was not changed in notebok 03

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("../data/interim/02_adata_parsedPerturb.h5ad")
print(adata)

In [ ]:
# Build control baseline once per gene, then flag each target cell as
# "responder" (own target-gene expression > control mean) or not.
# Restricted to single-gene targets only -- doubles would need baseline
# comparisons for two genes at once, out of scope for this simple approach.

adata.obs["is_responder"] = False  # default; only single-target cells get evaluated

# here, .loc both matches idx based on the "single" mask, and then pulls the "target" column
single_targets = adata.obs.loc[adata.obs["perturbation_type"] == "single", "target"].unique()
is_control = (adata.obs["target"] == "control").values

responder_flags = pd.Series(False, index=adata.obs_names)

for gene in single_targets:
    if gene not in adata.var_names:
        continue
    idx = adata.var_names.get_loc(gene)
    expr = np.asarray(adata.layers["lognorm"][:, idx].todense()).flatten()
    control_mean = expr[is_control].mean()

    is_target = (adata.obs["target"] == gene).values
    # TRUE for cell is responder, if value is above control_mean
    responder_flags[is_target] = expr[is_target] > control_mean

adata.obs["is_responder"] = responder_flags.values

print(adata.obs.loc[adata.obs["perturbation_type"] == "single", "is_responder"].value_counts())

In [ ]:
# Restrict to responder cells + controls, for DE input

de_adata = adata[
    (adata.obs["is_responder"]) | (adata.obs["target"] == "control")
].copy()

print(de_adata.obs["target"].value_counts().loc[
    de_adata.obs["target"].value_counts().index.isin(list(single_targets) + ["control"])
].head())
print(de_adata.shape)

In [ ]:
# when we built "target" col in 02 and cast it, had all 105 single-gene names, and all the "GENE1+GENE2" double-perturbation combos.
# categorical col stores per-row values and cat.categories (list of all possible categories)
# when we dropped cells from adata to de_adata, the values in "target" shrank but cat.categories didn't remove possiblities

# remove unused categories in "target" (not retained in filtering)
de_adata.obs["target"] = de_adata.obs["target"].cat.remove_unused_categories()

# print first 5 targets, retained, and total # targets survived filtering
print(de_adata.obs["target"].cat.categories.tolist()[:5], "...", len(de_adata.obs["target"].cat.categories), "total categories")

In [ ]:
# use scanpy to run DE. sc.tl is "tools"
# this is written to de_data.uns["rank_genes_groups"], which is a dict of structured arrays. not a new column or separate object; stored per-group.
# after run, this: print(de_adata.uns["rank_genes_groups"].keys()) has names, scores, pvals, logfoldchanges, pvals_adj

# here, using Wilcoxon rank-sum test (Mann-Whitney U) at the single cel level
# For each gene, ranks all cells (target and ctrl) by that genes expression, and tests whether one group systematically DE
# this is NOT psuedobulk. you would collapse cells, but leave "lanes" separate as replicates
# pitfall with Wilcoxon here is that it treates ever single cell as an independent data point, but they are not independent wince they came from the same dish/treatment
# this inflates statistical power artificially (same issue we saw using Mann-Whitney U and getting 75/100 target genes p = 0)

# L2FC is reported with this, but computed separately by Scanpy comparing mean expression per group

sc.tl.rank_genes_groups(
    de_adata,
    groupby="target",
    groups=[g for g in de_adata.obs["target"].cat.categories if g != "control"],
    reference="control",
    method="wilcoxon",
    layer="lognorm",
    use_raw=False
)

In [ ]:
# sc.get is a convenience fxn for extracting things that tools generated and buried in AnnData object
# here we get the de results and convert to pd df

def get_de_results(de_adata, group, n_genes=None):
    """Extract DE results for one target as a clean DataFrame."""
    result = sc.get.rank_genes_groups_df(de_adata, group=group)
    if n_genes:
        result = result.head(n_genes)
    return result

# quick look at one target's top downstream DE genes
example = get_de_results(de_adata, "KLF1", n_genes=15)
example

# KLF1 is a TF - master regulator of erythroid differentiation in K562 cells
# a lot of the top genes here (HBZ, HBG2, HBA1, HBG1, BLVRB, ANXA2, S100A10) are known erythroid markers
# this is a good validation the experiment and analysis worked: downstream (of KLF1 perturbation) DEGs make sense
# note p-vals are inflated in significance (too low) - because of Wilcoxon/Mann-Whitney U and sample size.  L2FC is still trustworthy


In [ ]:
# save adata
# new: is_responder and .uns are now attached

adata.write("../data/interim/04_adata_responder_flagged.h5ad", compression="gzip")

In [ ]:
# Export all 105 targets' Wilcoxon DE results as one tidy CSV
# (loop sc.get.rank_genes_groups_df across every group and concatenate, rather than leaving results locked inside the .uns blob):

all_de = []
for target in de_adata.obs["target"].cat.categories:
    if target == "control":
        continue
    df = sc.get.rank_genes_groups_df(de_adata, group=target)
    df["target"] = target
    all_de.append(df)

wilcoxon_de = pd.concat(all_de, ignore_index=True)
wilcoxon_de.to_csv("../data/interim/04_wilcoxon_de_results.csv.gz", index=False)
print(wilcoxon_de.shape)
wilcoxon_de.head()